# 16 · RNRA-M: plasticidad multiescala (una inteligencia, varios tamaños)

**Estado: sin validar.** Rama del proyecto 15, evaluada aparte. Una sola red que cambia de tamaño según el dispositivo **sin destruir lo que sabe**:

- **Núcleo persistente:** las primeras neuronas de cada capa, cuantizadas, es lo único que ocupa RAM en el celular.
- **Capacidad expandible:** los anchos 32 → 128 → 512 comparten pesos (superred anidada, como las redes *slimmable* y Once-for-All).
- **Memoria portátil:** lo recortado más el residuo de cuantizar el núcleo viajan **comprimidos en el mismo archivo** (`.npz`, sin `pickle`). No hace falta internet: la inteligencia se copia completa y al volver a la PC se reconstruye la red grande.

**Hipótesis (deben poder refutarse)**
- **H1** una red entrenada para varios presupuestos rinde mejor que una comprimida después de entrenar.
- **H2** una memoria de adaptaciones permite recuperar capacidades al expandirse.

**Qué se mide y qué no:** precisión, RAM **calculada** (bytes de pesos, no medida en un teléfono), latencia en la CPU de Colab (no del aparato real), capacidad recuperada al expandir. Energía y RAM reales requieren el dispositivo.

**Antecedentes que hay que superar:** Once-for-All, Slimmable Networks, destilación, LoRA y cuantización. Cambiar de tamaño no es nuevo; lo que se prueba es combinarlo con una memoria recuperable.

**Qué se quitó:** el experimento de aprender en el celular y volver a la PC (H3) no aportó nada medible frente a repasar un recuerdo de la tarea vieja; está en el commit `fd6e8f3`.

## 1. Preparar (≈ 2 min)

In [ ]:
import os, subprocess
from google.colab import userdata

if not os.path.exists("/content/Notbook"):
    token = userdata.get("GITHUB_TOKEN")
    subprocess.run(["git", "clone", "--depth", "1", f"https://{token}@github.com/SiriusMagic/Notbook.git", "/content/Notbook"], check=True)
%cd /content/Notbook
!pip install -q -r laboratorio/requirements.txt

In [ ]:
import numpy as np, torch, matplotlib.pyplot as plt
from laboratorio.banco.tareas import cargar_tarea
from laboratorio.rnra import experimentos_m as em
from laboratorio.rnra import memoria_multiescala as mem
from laboratorio.rnra.adaptador_dispositivo import dispositivos, elegir
from laboratorio.rnra.superred import ANCHOS, SuperRed, bytes_modelo, entrenar, exactitud

SEMILLAS = range(5)
PASOS = 400
print("Anchos:", ANCHOS)

## 2. Reducir y expandir (≈ 6 min en CPU)

Para cada dispositivo (celular básico = cabe el ancho 32 en 32 bits; potente = 128; PC = 512): red fija por dispositivo, red de 128 entrenada sola y cuantizada después, superred de ancho variable y RNRA-M. Después: ir al núcleo pequeño y **volver** al grande, con memoria (32, 8 y 4 bits) y sin ella.

In [ ]:
t = cargar_tarea("caotica_20d")
res1 = [em.reducir_expandir(t, s, pasos=PASOS) for s in SEMILLAS]
print(em.tablas_reducir_expandir(res1))

### Cuánto pesa cada pieza

El núcleo es lo único en RAM; la memoria es almacenamiento. El archivo completo puede pesar tanto como la red grande: **se ahorra RAM de trabajo, no espacio en disco**.

In [ ]:
sup = entrenar(SuperRed(t.entradas, t.salidas), t.x_entreno, t.y_entreno, pasos=PASOS)
nombres, valores = [], []
for w in ANCHOS:
    for b in (32, 8):
        nombres.append(f"{w} neuronas, {b} bits"); valores.append(bytes_modelo(sup, w, b) / 1024)
plt.barh(nombres, valores); plt.xscale("log"); plt.xlabel("KB de pesos (escala logarítmica)"); plt.show()
for bm in (32, 8, 4):
    p = mem.reducir(sup, ANCHOS[0], 8, bm)
    print(f"memoria de {bm} bits: núcleo {p.bytes_nucleo()/1024:.1f} KB en RAM + memoria {p.bytes_memoria()/1024:.0f} KB comprimida")

## 3. Lectura honesta

- Si «reducir y expandir sin memoria» ya da casi lo mismo que la red original, la tarea no exige capacidad y la memoria no se nota: hace falta una tarea más difícil (Fashion-MNIST, texto).
- El ahorro de RAM lo da la cuantización a 8 bits (`fija_int8` ahorra lo mismo); lo propio de RNRA-M es que **una sola red** sirve para los tres dispositivos y que la memoria devuelve la red grande idéntica.
- La pregunta pendiente es el tamaño real: con millones de parámetros, la memoria comprimida y la cuantización por canal importan mucho más que aquí.

Los umbrales y las cifras medidas están en `laboratorio/banco/informes/16-plasticidad-multiescala.md`.

## 4. Guardar el informe en Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/Notbook/resultados_rnra", exist_ok=True)
!python -m laboratorio.rnra.experimentos_m --semillas 5 --pasos 400 --tarea caotica_20d --salida /content/drive/MyDrive/Notbook/resultados_rnra/16-plasticidad-multiescala.md > /dev/null
print("Guardado en Drive: Notbook/resultados_rnra/")